# Feature Engineering

In [23]:
#importando bibliotecas
import pandas as pd
import plotly.express as px

import pandas as pd
from sklearn.cluster import KMeans
from kneed import KneeLocator

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

pd.set_option('display.max_columns', None)

In [24]:
def find_best_k(scaled_data, max_k=10):
    k_range = range(1, max_k + 1)
    sse = []
    
    for i in k_range:
        kmeans = KMeans(n_clusters=i, random_state=42, n_init=10)
        kmeans.fit(scaled_data)
        sse.append(kmeans.inertia_)
    
    # Identifica o "joelho" (elbow) na curva de SSE
    kl = KneeLocator(k_range, sse, curve='convex', direction='decreasing')
    return kl.elbow

In [25]:
#importando dados
df = pd.read_pickle('../data/curated/data.pkl')

Definindo perfil do aeroporto de origem

In [26]:
df_origin_airports_profile = df.groupby('ORIGIN_AIRPORT').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()

df_origin_airports_profile.columns = ['ORIGIN_AIRPORT', 'DELAY_RATE', 'FLIGHT_VOLUME']

scaler = StandardScaler()
features_to_scale = ['DELAY_RATE', 'FLIGHT_VOLUME']
df_scaled = scaler.fit_transform(df_origin_airports_profile[features_to_scale])

n_clusters = find_best_k(df_scaled) 

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_origin_airports_profile['ORIGIN_AIRPORT_PROFILE'] = kmeans.fit_predict(df_scaled)

df_origin_airports_profile

,ORIGIN_AIRPORT,DELAY_RATE,FLIGHT_VOLUME,ORIGIN_AIRPORT_PROFILE
0,ABE,0.171980,2227,0
1,ABI,0.153294,2231,0
2,ABQ,0.172957,18918,0
3,ABR,0.173454,663,0
4,ABY,0.173210,866,0
...,...,...,...,...
314,WRG,0.164869,649,0
315,WYS,0.067308,208,2
316,XNA,0.219346,8963,0
317,YAK,0.109231,650,2


In [27]:
fig = px.scatter(
    df_origin_airports_profile, 
    x='FLIGHT_VOLUME', 
    y='DELAY_RATE',
    color='ORIGIN_AIRPORT_PROFILE',
    hover_name='ORIGIN_AIRPORT',
    title='Agrupamento de Aeroportos: Volume vs. Taxa de Atraso',
    labels={'FLIGHT_VOLUME': 'Volume Total de Voos', 'DELAY_RATE': 'Taxa de Atraso (0 a 1)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.show()

Os aeroportos localizados à extrema direita representam os grandes hubs, que, apesar do altíssimo volume de tráfego, mantêm taxas de atraso moderadas e consistentes. Já no lado esquerdo, onde se concentra a maioria dos aeroportos, o algoritmo diferenciou com sucesso os grupos por performance: o cluster superior (em amarelo) identifica aeroportos de baixo volume mas com alta instabilidade (altos atrasos), enquanto o cluster inferior (em laranja) destaca os aeroportos menores e mais pontuais.

Definindo perfil do aeroporto de destino

In [28]:
df_dest_airports_profile = df.groupby('DESTINATION_AIRPORT').agg({
'IS_DELAYED': ['mean', 'count']
}).reset_index()

df_dest_airports_profile.columns = ['DESTINATION_AIRPORT', 'DELAY_RATE', 'FLIGHT_VOLUME']

scaler = StandardScaler()
features_to_scale = ['DELAY_RATE', 'FLIGHT_VOLUME']
df_scaled = scaler.fit_transform(df_dest_airports_profile[features_to_scale])

n_clusters = find_best_k(df_scaled)


kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_dest_airports_profile['DESTINATION_AIRPORT_PROFILE'] = kmeans.fit_predict(df_scaled)

df_dest_airports_profile

,DESTINATION_AIRPORT,DELAY_RATE,FLIGHT_VOLUME,DESTINATION_AIRPORT_PROFILE
0,ABE,0.182883,2220,2
1,ABI,0.181655,2224,2
2,ABQ,0.198491,18953,2
3,ABR,0.106545,657,0
4,ABY,0.215278,864,2
...,...,...,...,...
314,WRG,0.197853,652,2
315,WYS,0.072464,207,0
316,XNA,0.227354,8986,2
317,YAK,0.144172,652,0


In [29]:
fig = px.scatter(
    df_dest_airports_profile, 
    x='FLIGHT_VOLUME', 
    y='DELAY_RATE',
    color='DESTINATION_AIRPORT_PROFILE',
    hover_name='DESTINATION_AIRPORT',
    title='Agrupamento de Aeroportos: Volume vs. Taxa de Atraso',
    labels={'FLIGHT_VOLUME': 'Volume Total de Voos', 'DELAY_RATE': 'Taxa de Atraso (0 a 1)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.show()

Definindo perfil da rota

In [30]:
df['ROUTE'] = df['ORIGIN_AIRPORT'] + '_' + df['DESTINATION_AIRPORT']

df_route_profile = df.groupby('ROUTE').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()

df_route_profile.columns = ['ROUTE', 'DELAY_RATE', 'FLIGHT_VOLUME']

scaler = StandardScaler()
features_to_scale = ['DELAY_RATE', 'FLIGHT_VOLUME']
df_scaled = scaler.fit_transform(df_route_profile[features_to_scale])

n_clusters = find_best_k(df_scaled)

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_route_profile['ROUTE_PROFILE'] = kmeans.fit_predict(df_scaled)

df_route_profile

,ROUTE,DELAY_RATE,FLIGHT_VOLUME,ROUTE_PROFILE
0,ABE_ATL,0.148984,886,2
1,ABE_DTW,0.179856,695,2
2,ABE_ORD,0.195046,646,2
3,ABI_DFW,0.153294,2231,2
4,ABQ_ATL,0.090113,799,2
...,...,...,...,...
4629,XNA_SFO,0.254902,51,0
4630,XNA_SLC,0.000000,1,2
4631,YAK_CDV,0.129231,325,2
4632,YAK_JNU,0.089231,325,2


In [31]:
fig = px.scatter(
    df_route_profile, 
    x='FLIGHT_VOLUME', 
    y='DELAY_RATE',
    color='ROUTE_PROFILE',
    hover_name='ROUTE',
    title='Agrupamento de Aeroportos: Volume vs. Taxa de Atraso',
    labels={'FLIGHT_VOLUME': 'Volume Total de Voos', 'DELAY_RATE': 'Taxa de Atraso (0 a 1)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.show()

Definindo perfil da companhia aérea

In [32]:
df_airline_profile = df.groupby('AIRLINE').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()

df_airline_profile.columns = ['AIRLINE', 'DELAY_RATE', 'FLIGHT_VOLUME']

scaler = StandardScaler()
features_to_scale = ['DELAY_RATE', 'FLIGHT_VOLUME']
df_scaled = scaler.fit_transform(df_airline_profile[features_to_scale])

n_clusters = find_best_k(df_scaled)

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_airline_profile['AIRLINE_PROFILE'] = kmeans.fit_predict(df_scaled)

df_airline_profile

,AIRLINE,DELAY_RATE,FLIGHT_VOLUME,AIRLINE_PROFILE
0,AA,0.181512,636554,1
1,AS,0.124916,157025,2
2,B6,0.222239,240304,3
3,DL,0.135375,791067,1
4,EV,0.194246,508288,3
5,F9,0.264786,81715,0
6,HA,0.107269,69815,2
7,MQ,0.219465,257130,3
8,NK,0.295904,104500,0
9,OO,0.184571,528328,3


In [33]:
fig = px.scatter(
    df_airline_profile, 
    x='FLIGHT_VOLUME', 
    y='DELAY_RATE',
    color='AIRLINE_PROFILE',
    hover_name='AIRLINE',
    title='Agrupamento de Aeroportos: Volume vs. Taxa de Atraso',
    labels={'FLIGHT_VOLUME': 'Volume Total de Voos', 'DELAY_RATE': 'Taxa de Atraso (0 a 1)'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig.show()

Obtendo período do dia

In [34]:
def get_time_of_day(raw):
    hour = raw // 100
    if 0 <= hour < 6:
        return 'OVERNIGHT'
    elif 6 <= hour < 12:
        return 'MORNING'
    elif 12 <= hour < 18:
        return 'AFTERNOON'
    else:
        return 'EVENING'

df['TIME_OF_DAY'] = df['SCHEDULED_DEPARTURE'].apply(get_time_of_day)

Obtendo estação do ano

In [35]:
seasons = {
    12: 'SUMMER', 1: 'SUMMER', 2: 'SUMMER',
    3: 'AUTUMN', 4: 'AUTUMN', 5: 'AUTUMN',
    6: 'WINTER', 7: 'WINTER', 8: 'WINTER',
    9: 'SPRING', 10: 'SPRING', 11: 'SPRING'
}

df['SEASON'] = df['MONTH'].map(seasons)

Obtendo horário programado de saída e chegada do voo

In [36]:
df['SCHEDULED_DEPARTURE_HOUR'] = df['SCHEDULED_DEPARTURE'] // 100
df['SCHEDULED_ARRIVAL_HOUR'] = df['SCHEDULED_ARRIVAL'] // 100

Mergeando os dados

In [37]:
df = df.merge(df_origin_airports_profile[['ORIGIN_AIRPORT', 'ORIGIN_AIRPORT_PROFILE']], on='ORIGIN_AIRPORT', how='left')
df = df.merge(df_dest_airports_profile[['DESTINATION_AIRPORT', 'DESTINATION_AIRPORT_PROFILE']], on='DESTINATION_AIRPORT', how='left')
df = df.merge(df_airline_profile[['AIRLINE', 'AIRLINE_PROFILE']], on='AIRLINE', how='left')
df = df.merge(df_route_profile[['ROUTE', 'ROUTE_PROFILE']], on='ROUTE', how='left')

Selecionando features

In [38]:
cols = [
    'MONTH', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE_HOUR', 'SCHEDULED_ARRIVAL_HOUR', 'SCHEDULED_TIME', 'TIME_OF_DAY', 
    'SEASON', 'DISTANCE', 'ORIGIN_AIRPORT_PROFILE', 'DESTINATION_AIRPORT_PROFILE', 'AIRLINE_PROFILE', 
    'ROUTE_PROFILE', 'IS_DELAYED'
]
df = df[cols]
df.head()

,MONTH,DAY_OF_WEEK,SCHEDULED_DEPARTURE_HOUR,SCHEDULED_ARRIVAL_HOUR,SCHEDULED_TIME,TIME_OF_DAY,SEASON,DISTANCE,ORIGIN_AIRPORT_PROFILE,DESTINATION_AIRPORT_PROFILE,AIRLINE_PROFILE,ROUTE_PROFILE,IS_DELAYED
0,1,4,0,4,205.0,OVERNIGHT,SUMMER,1448,2,1,2,1,0
1,1,4,0,7,280.0,OVERNIGHT,SUMMER,2330,1,2,1,0,0
2,1,4,0,8,286.0,OVERNIGHT,SUMMER,2296,1,1,3,2,0
3,1,4,0,8,285.0,OVERNIGHT,SUMMER,2342,1,2,1,1,0
4,1,4,0,3,235.0,OVERNIGHT,SUMMER,1448,1,0,2,1,0


Obtendo correlações

In [39]:
df_corr = df.copy()

seasons = [['WINTER', 'SPRING', 'AUTUMN', 'SUMMER']]
tome_of_day = [['OVERNIGHT', 'MORNING', 'AFTERNOON', 'EVENING']]

encoded_seasons = OrdinalEncoder(categories=seasons)
encoded_time_or_day = OrdinalEncoder(categories=tome_of_day)

df_corr['SEASON_NUM'] = encoded_seasons.fit_transform(df_corr[['SEASON']])
df_corr['TIME_OF_DAY_NUM'] = encoded_time_or_day.fit_transform(df_corr[['TIME_OF_DAY']])

num_cols = [
    'MONTH', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE_HOUR', 'SCHEDULED_ARRIVAL_HOUR', 'SCHEDULED_TIME',
    'TIME_OF_DAY_NUM', 'SEASON_NUM', 'DISTANCE', 'ORIGIN_AIRPORT_PROFILE', 'DESTINATION_AIRPORT_PROFILE', 
    'ROUTE_PROFILE', 'IS_DELAYED'
]

df_corr = df_corr[num_cols].corr()

In [40]:
fig = px.imshow(
    df_corr,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu_r',
    zmin=-1, 
    zmax=1,
    title='Matriz de Correlação',
    labels=dict(color='Correlação')
)
fig.update_layout(
    height=600,
    template='plotly_white'
)
fig.show()

Balanceando target

In [41]:
target = 'IS_DELAYED'

df_atrasados = df[df[target] == 1]
df_pontuais = df[df[target] == 0]

n_atrasados = len(df_atrasados)
df_pontuais_bal = df_pontuais.sample(n=n_atrasados, random_state=42)

df_balanced = pd.concat([df_atrasados, df_pontuais_bal])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

Normalizando features

In [42]:
cols_categoricas = ['SEASON', 'TIME_OF_DAY']
cols_numericas = [
    'DISTANCE', 'MONTH', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE_HOUR', 'SCHEDULED_ARRIVAL_HOUR', 'SCHEDULED_TIME', 'ORIGIN_AIRPORT_PROFILE',
    'DESTINATION_AIRPORT_PROFILE', 'AIRLINE_PROFILE', 'ROUTE_PROFILE',
]

df_normalized = df_balanced.copy()

scaler = StandardScaler()
df_normalized[cols_numericas] = scaler.fit_transform(df_normalized[cols_numericas])

df_normalized = pd.get_dummies(df_normalized, columns=cols_categoricas, dtype=int)

df_normalized.head()

,MONTH,DAY_OF_WEEK,SCHEDULED_DEPARTURE_HOUR,SCHEDULED_ARRIVAL_HOUR,SCHEDULED_TIME,DISTANCE,ORIGIN_AIRPORT_PROFILE,DESTINATION_AIRPORT_PROFILE,AIRLINE_PROFILE,ROUTE_PROFILE,IS_DELAYED,SEASON_AUTUMN,SEASON_SPRING,SEASON_SUMMER,SEASON_WINTER,TIME_OF_DAY_AFTERNOON,TIME_OF_DAY_EVENING,TIME_OF_DAY_MORNING,TIME_OF_DAY_OVERNIGHT
0,0.253167,-1.456333,1.135593,1.143697,1.624644,1.580078,0.461064,1.202951,-0.848948,-1.500206,1,0,0,0,1,0,1,0,0
1,1.436686,-0.953642,0.716535,0.557100,-1.134818,-1.216751,-1.241973,-0.516591,1.077930,1.222706,1,0,1,0,0,1,0,0,0
2,-1.522113,0.554430,1.135593,0.948165,-1.041952,-1.014511,0.461064,1.202951,1.077930,-1.500206,1,0,0,1,0,0,1,0,0
3,-1.226233,1.559811,-0.121581,-0.029496,0.112246,0.003265,0.461064,1.202951,1.077930,1.222706,0,0,0,1,0,1,0,0,0
4,1.436686,1.559811,-0.331110,-0.029496,0.311246,0.049304,-1.241973,-0.516591,-0.848948,1.222706,0,0,1,0,0,1,0,0,0


Exportando features

In [43]:
df_normalized.to_pickle('../data/curated/features.pkl')